# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one content page (content_id), belonging to one client
(client_id), summarized over a fixed 90-day observation window (the
column suffix _90d confirms this -- e.g. impressions_90d, sessions_90d).
I verify this below by checking that content_id is unique per row.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DataWithHamza/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "reportlab"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/Flyrank-ML-Internship
Starter data found. You're ready.


## 2. Fields: feature / label / context / excluded

Feature (observed signals, safe to use as model inputs):
- search_volume, competition, cpc, word_count, char_count
- impressions_90d, clicks_90d, sessions_90d, ai_sessions_90d
- days_with_impressions, days_with_sessions, content_age_days
- ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct
- competition_level, content_type, main_intent (categorical)

Derived/context (tiers -- fine as context, or as engineered features,
but transparent so I can explain them):
- age_tier, freshness_tier, word_count_tier, impression_tier,
  position_tier, trend_pct, days_since_last_update

Label/proxy:
- trend_direction (specifically is_declining = trend_direction == "down")
  -- this is a current-window proxy label, not a future outcome (flagged
  in the lane guide as a known limitation).

Excluded:
- client_id, content_id -- join/grouping keys only, never model features
  (they carry no real signal, just identity).
- Any product decision flags (health_score, priority_score, action_type)
  -- not present in this dataset by design, so nothing to exclude here,
  but noting I will never rebuild and feed one back in as a feature.

In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total rows:", df.shape[0])
print("Unique content_id:", df["content_id"].nunique())
print("Row == unique content_id?", df.shape[0] == df["content_id"].nunique())
print("Unique client_id (clients in this slice):", df["client_id"].nunique())

Total rows: 30000
Unique content_id: 30000
Row == unique content_id? True
Unique client_id (clients in this slice): 32


In [3]:
feature_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
                "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
                "days_with_impressions", "days_with_sessions", "content_age_days",
                "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
                "competition_level", "content_type", "main_intent"]

label_col = "trend_direction"
excluded_cols = ["client_id", "content_id"]

present_features = [c for c in feature_cols if c in df.columns]
missing_features = [c for c in feature_cols if c not in df.columns]

print("Feature columns present:", present_features)
print("Feature columns NOT found (check spelling vs data dictionary):", missing_features)
print("Label column present:", label_col in df.columns)

Feature columns present: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent']
Feature columns NOT found (check spelling vs data dictionary): []
Label column present: True


## 3. Verify it with queries (grain, counts, missing values, windows)

Verified: content_id has zero duplicates (grain claim holds). Missing
values are concentrated in [name columns once you see the output] -- I
will decide feature-by-feature whether to impute or drop. The pipeline's
own filters (impressions_90d > 0, content_age_days >= 90) already
narrow the dataset before modeling, which I've reproduced above to
confirm row counts match what scripts/01_prepare_features.py reports.

In [4]:
# Grain check
print("Duplicate content_id rows:", df["content_id"].duplicated().sum())

# Missing values on key columns
key_cols = present_features + [label_col]
print("\nMissing values per key column:")
print(df[key_cols].isna().sum().sort_values(ascending=False))

# Window check: confirm the 90-day filter the pipeline applies
print("\nRows with impressions_90d > 0:", (df["impressions_90d"] > 0).sum())
print("Rows with content_age_days >= 90:", (df["content_age_days"] >= 90).sum())
print("Rows passing BOTH filters (matches pipeline's prepare step):",
      ((df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)).sum())

# Label distribution
print("\ntrend_direction value counts:")
print(df["trend_direction"].value_counts())

Duplicate content_id rows: 0

Missing values per key column:
char_count               7699
word_count               7699
competition_level        2610
cpc                      2468
search_volume            2468
competition              2468
main_intent              2374
scroll_rate               125
impressions_90d             0
days_with_impressions       0
clicks_90d                  0
sessions_90d                0
ai_sessions_90d             0
ctr                         0
content_age_days            0
days_with_sessions          0
avg_position                0
ai_traffic_pct              0
engagement_rate             0
content_type                0
trend_direction             0
dtype: int64

Rows with impressions_90d > 0: 30000
Rows with content_age_days >= 90: 30000
Rows passing BOTH filters (matches pipeline's prepare step): 30000

trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int6

## 4. Data limits

This starter dataset can never tell me:
1. Causal impact -- it can show a page declined, never that a specific
   fix caused a recovery. That needs a real experiment.
2. True future outcomes -- trend_direction is a current-window bucket,
   not a next-30-days observed result. My label is a proxy, not ground
   truth about the future.
3. Full history depth -- per the lane guide, client history is an
   unbalanced panel (different clients have different amounts of
   tracking history); this starter slice is a fixed 30,000-row sample,
   not the full warehouse, so I can't yet check per-client history
   coverage (gsc_data_start / ga4_data_start) -- that only exists in the
   full warehouse release.
4. Pre-GA4 rows would only have search data, no session/engagement data
   -- I don't yet know if this starter slice includes any such rows;
   I'll check missing values in sessions_90d / engagement_rate for
   that pattern (see Section 3 output).
5. Any raw identity -- client_id and content_id are pseudonymized hashes;
   I can group and join with them, but they carry zero real-world meaning.